# Productivización de modelos

Quizás uno de los aspectos clave es cómo poner en valor los modelos construidos para que tengan impacto en los procesos de negocio. Existen distintas modalidades en las que este proceso toma forma. Disponer de un entorno con garantías de qué modelo es el correcto a poner en marcha es quizás una de las claves a la hora de dar servicio a escala en la mayoría de las organizaciones. Veremos formas _manuales_ de hacerlo, pero es bueno que conozcamos las mejores prácticas en lo que respecta al servicio de modelos o _model serving_

En la actualidad muchas de estas plataformas se han especializado en dos modalidades, ML y Gen AI.


## MLFlow

Ampliaremos el ejercicio anteriormente realizado con Comet para el caso de MLFlow desplegado de forma local. MLFlow nos permite desplegar un servicio y actuar de forma local incluyendo el poder servir un modelo registrado en nuestro servidor de experimentos.

* https://mlflow.org/docs/latest/introduction/index.html

Una vez instalado podemos ejecutar nuestro servidor para que se quede "escuchando" en el puerto 5000. Deberemos abrir un terminal con el entorno python donde instalamos mlflow activo y ejecutar:

```sh
mlflow ui
```

No cerréis el terminal ya que el proceso se cerrará. Podéis acceder a la ruta http://127.0.0.1:5000/ para acceder a la interfaz local de vuestro sistema. Esto os permite configurar vuestro entorno Python para que emplee este registro como el punto en el que registrar nuestras métricas y modelos.

In [ ]:
# %pip install mlflow

  Using cached mlflow-3.12.0-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-3.12.0-py3-none-any.whl.metadata (50 kB)
  Using cached mlflow_tracing-3.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached aiohttp-3.13.5-cp311-cp311-macosx_11_0_arm64.whl.metadata (8.1 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.7-cp311-abi3-macosx_10_9_universal2.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached gunicorn-25.3.0-py3-none-any.whl.metadata (5.5 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached pandas-2.3.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pyarrow-23.0.1-cp311-cp311-macosx_12_0_arm64.whl.metadata (3.1 kB)
  Using c

In [1]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

Al igual que hicimos con Comet, podemos registrar las métricas que creamos relevantes para un experimento.

In [8]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
# Verificar que apunta al servidor correcto:
print(mlflow.get_tracking_uri())  # Debe mostrar http://127.0.0.1:5000

http://127.0.0.1:5000


In [5]:
import mlflow
import os

# Apunta explícitamente a la misma DB que mlflow ui está usando
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [6]:
mlflow.set_experiment("check-localhost-connection")

with mlflow.start_run():
    mlflow.log_metric("foo", 1)
    mlflow.log_metric("bar", 2)

Volver al interfaz para ver cómo un nuevo experimento fue registrado y las métricas asociadas a este. Veréis que no hay mucha magia ya que los datos como tal se registran en una carpeta en la ruta en la que estamos trabajando (revisad las carpetas _mlruns_ y _mlartifacts_).

In [11]:
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment("mi-experimento-rf")

with mlflow.start_run() as run:
    X, y = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    params = {"max_depth": 2, "random_state": 42}
    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    # Log parameters and metrics using the MLflow APIs
    mlflow.log_params(params)

    y_pred = model.predict(X_test)
    mlflow.log_metrics({"mse": mean_squared_error(y_test, y_pred)})

    # Log the sklearn model and register as version 1
    mlflow.sklearn.log_model(
        sk_model=model,
        name="sklearn-model",
        input_example=X_train,
        registered_model_name="sk-learn-random-forest-reg-model",
    )

2026/05/26 12:57:57 INFO mlflow.tracking.fluent: Experiment with name 'mi-experimento-rf' does not exist. Creating a new experiment.
2026/05/26 12:57:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'sk-learn-random-forest-reg-model'.
2026/05/26 12:58:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: sk-learn-random-forest-reg-model, version 1


🏃 View run resilient-crow-316 at: http://127.0.0.1:5000/#/experiments/1/runs/5b3ead16968c46568cfcd9de0c63b7cc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '1' of model 'sk-learn-random-forest-reg-model'.


Acabamos de registrar nuestro primer modelo http://127.0.0.1:5000/#/models/sk-learn-random-forest-reg-model. Podemos incluir información adicional (etiquetas) para conocer de qué tipo de modelo se trata.

![modelo](https://mlflow.org/docs/latest/assets/images/model-alias-and-tags-0318d486b2bf16992f488de5a00ce474.png)

Cualquier modelo registrado es accesible una vez tenemos el servidor de MLFlow en marcha. De este modo podemos rescatar distintas versiones del modelo de una forma centralizada.

In [12]:
import mlflow.sklearn
from sklearn.datasets import make_regression

model_name = "sk-learn-random-forest-reg-model"
model_version = "1"

# Load the model from the Model Registry
model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.sklearn.load_model(model_uri)

# Generate a new dataset for prediction and predict
X_new, _ = make_regression(n_features=4, n_informative=2, random_state=0, shuffle=False)
y_pred_new = model.predict(X_new)

print(y_pred_new)

[ 16.36355607 -20.09258424   8.0136586    6.16919118  -1.81185423
   4.03116362 -24.95801449  68.78053495 -45.0766513   64.44760141
 -40.16931792 -25.54191065 -14.39985794 -38.0567874    8.05358765
 -25.73029816 -15.91990041 -10.99985266 -24.2475118  -32.70582446
  17.34781751  68.78053495  44.27341488  41.31593646  48.16602726
 -23.62019943  47.15590018  69.12741949  48.16602726  -0.26024544
 -28.49126919 -10.99985266  10.73067585 -10.61092056  -4.7324722
   2.76556278  58.93099448 -31.19567455 -35.55773052 -23.99366895
  48.16602726  13.34984948  12.56552213 -18.66808469 -32.70582446
 -39.30386685 -34.29680647  48.16602726 -33.40149961  20.35083862
 -15.0214084  -34.55064932  -2.28963784 -19.61227378   7.6979477
 -25.86538741 -11.95702358 -15.36598686   5.88539811 -30.23881739
 -25.47645531 -43.61170248 -43.7442754  -14.59055495 -40.16931792
 -32.70582446  -2.68114572  -5.39418041  16.15991316  -2.28963784
  41.662821    10.04512765  51.22797543 -23.09874036  10.04512765
  46.5774364

## Ejemplo completo

Nuestro data scientist procede a obtener los datos y realizar su magia encontrando un modelo que devuelve buenos resultados.

In [13]:
import pandas as pd
from mlflow.models import infer_signature

# Load dataset
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)

# Split the data into training, validation, and test sets
train, test = train_test_split(data, test_size=0.25, random_state=42)
train_x = train.drop(["quality"], axis=1).values
train_y = train[["quality"]].values.ravel()
test_x = test.drop(["quality"], axis=1).values
test_y = test[["quality"]].values.ravel()
train_x, valid_x, train_y, valid_y = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)
signature = infer_signature(train_x, train_y)

[Hyperopt](https://hyperopt.github.io/hyperopt/) es una alternativa a otros sistemas de búsqueda de hiperparámetros. Nos permite buscar una serie de hiperparámetros para nuestro modelo de forma eficiente y distribuida. Esto se vuelve muy importante cuando requerimos entrenar modelo pesado como las redes neuronales a escala.

In [ ]:
# %pip install hyperopt
%pip install -U git+https://github.com/hyperopt/hyperopt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/hyperopt/hyperopt to /private/var/folders/tw/q38f_br56j93gr_g2pvmv9_m0000gn/T/pip-req-build-1bm0uc5g
  Running command git clone --filter=blob:none --quiet https://github.com/hyperopt/hyperopt /private/var/folders/tw/q38f_br56j93gr_g2pvmv9_m0000gn/T/pip-req-build-1bm0uc5g
  Resolved https://github.com/hyperopt/hyperopt to commit c49ad148201c81c6ad1b43730ef4d0912734aa37
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hyperopt: filename=hyperopt-0.3.0-py3-none-any.whl size=970867 sha256=556fe4cb324fc448ff583891fc2e4edb1f172122fdbdbb98def2158dae31d7ea
  Stored in directory: /private/var/folders/tw/q38f_br56j93gr_g2pvmv9_m0000gn/T/pip-ephem-wheel-cache-3vnj14hv/wheels/ad/e0/dc/af4d21315718e63bc2e53ded682ca11817b02cb72d

In [15]:
import keras
import numpy as np
from hyperopt import STATUS_OK

def train_model(params, epochs, train_x, train_y, valid_x, valid_y, test_x, test_y):
    # Define model architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)
    model = keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(1),
        ]
    )

    # Compile model
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"], momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()],
    )

    # Train model with MLflow tracking
    with mlflow.start_run(nested=True):
        model.fit(
            train_x,
            train_y,
            validation_data=(valid_x, valid_y),
            epochs=epochs,
            batch_size=64,
        )
        # Evaluate the model
        eval_result = model.evaluate(valid_x, valid_y, batch_size=64)
        eval_rmse = eval_result[1]

        # Log parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)

        # Log model
        mlflow.tensorflow.log_model(model, "model", signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

La función objetivo, como en todo proceso de optimización, guía cómo de bien estamos cambiando los parámetros de nuestro proceso. En este caso serán los hiperparámetros de nuestro entrenamiento (learning-rate y momentum).

In [16]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [17]:
from hyperopt import Trials, fmin, hp, tpe

space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0),
}

mlflow.set_experiment("wine-quality")

2026/05/26 12:59:21 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1779793161225, experiment_id='2', last_update_time=1779793161225, lifecycle_stage='active', name='wine-quality', tags={}, trace_location=None, workspace='default'>

In [18]:
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 12s 269ms/step - loss: 38.6272 - root_mean_squared_error: 6.2151
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 34.2267 - root_mean_squared_error: 5.8504 - val_loss: 31.7605 - val_root_mean_squared_error: 5.6356

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 30.4958 - root_mean_squared_error: 5.5223
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 947us/step - loss: 29.2638 - root_mean_squared_error: 5.4096 - val_loss: 27.1893 - val_root_mean_squared_error: 5.2143

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 28.5610 - root_mean_squared_error: 5.3443
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 853us/step - loss: 25.0258 - root_mean_squared_error: 5.0026 - val_loss: 23.2652 - val_root_mean_squared_error: 4.8234

 1/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 23.8715 - root_mean_squared_error: 4.8858
12/12 ━━━━━━━━━━━━━━━━━━━━ 

2026/05/26 12:59:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run shivering-finch-410 at: http://127.0.0.1:5000/#/experiments/2/runs/270f3cdf24304d7ab5714206150d4193

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

Epoch 1/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 11s 245ms/step - loss: 41.9658 - root_mean_squared_error: 6.4781
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 41.8822 - root_mean_squared_error: 6.4716 - val_loss: 38.8451 - val_root_mean_squared_error: 6.2326

Epoch 2/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 40.0894 - root_mean_squared_error: 6.3316
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 883us/step - loss: 34.8320 - root_mean_squared_error: 5.9019 - val_loss: 32.1509 - val_root_mean_squared_error: 5.6702

Epoch 3/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 32.2285 - root_mean_squared_error: 5.6770
46/46 ━━━━━━━━━━━━━

2026/05/26 12:59:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run painted-bug-249 at: http://127.0.0.1:5000/#/experiments/2/runs/7234af1c20504975bbc654dea4b32bbd

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                  

Epoch 1/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 12s 281ms/step - loss: 36.0342 - root_mean_squared_error: 6.0028
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 28.8812 - root_mean_squared_error: 5.3741 - val_loss: 21.0972 - val_root_mean_squared_error: 4.5932

Epoch 2/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 21.2799 - root_mean_squared_error: 4.6130
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 814us/step - loss: 15.7752 - root_mean_squared_error: 3.9718 - val_loss: 11.3646 - val_root_mean_squared_error: 3.3711

Epoch 3/3                                                                     

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 13.1862 - root_mean_squared_error: 3.6313
46/46

...

🏃 View run rare-cat-728 at: http://127.0.0.1:5000/#/experiments/2/runs/609fc9b2573343339ccc92178a9fb43a

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 11s 250ms/step - loss: 36.3751 - root_mean_squared_error: 6.0312
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 16.7114 - root_mean_squared_error: 4.0880 - val_loss: 6.2522 - val_root_mean_squared_error: 2.5004

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 7.1846 - root_mean_squared_error: 2.6804
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 830us/step - loss: 3.8188 - root_mean_squared_error: 1.9542 - val_loss: 2.5300 - val_root_mean_squared_error: 1.5906

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.1761 - root_mean_squared_error: 1.4751
46/46 ━━━

2026/05/26 12:59:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run handsome-rook-964 at: http://127.0.0.1:5000/#/experiments/2/runs/d7d18cc3c0ac464c8d78bcb8501579b6

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

100%|██████████| 8/8 [00:33<00:00,  4.17s/trial, best loss: 0.8991919755935669]

2026/05/26 12:59:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.006306536273585722), 'momentum': np.float64(0.6346348985938022)}
Best eval rmse: 0.8991919755935669
🏃 View run painted-finch-777 at: http://127.0.0.1:5000/#/experiments/2/runs/d26979399ec54623b2a9ffdc03cb1e78
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


Nuestro mejor RMSE es de 0.71 con los parámetros:

* learning-rate: 0.045
* momentum: 0.73

**NOTA**: Vuestro parámetros pueden variar ligeramente.

Verificad en el interfaz de MLFlow si esto es así. Podéis volver a ejecutar la celda y evaluar esta nueva ejecución.

In [19]:
mlflow.set_experiment("wine-quality")
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=8,
        trials=trials,
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 11s 257ms/step - loss: 31.5166 - root_mean_squared_error: 5.6140
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 18.5977 - root_mean_squared_error: 4.3125 - val_loss: 8.2084 - val_root_mean_squared_error: 2.8650

Epoch 2/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 9.0652 - root_mean_squared_error: 3.0108
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 858us/step - loss: 5.0135 - root_mean_squared_error: 2.2391 - val_loss: 3.2436 - val_root_mean_squared_error: 1.8010

Epoch 3/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.3073 - root_mean_squared_error: 1.8186
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 811us/step - loss: 2.8578 - root_mean_squared_error: 1.6905 - val_loss: 2.5048 - val_root_mean_squared_error: 1.5826

 1/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.1255 - root_mean_squared_error: 1.4579
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/st

2026/05/26 13:08:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run welcoming-yak-230 at: http://127.0.0.1:5000/#/experiments/2/runs/5ef892aed9a14a689ff1c0a298edf9e1

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 10s 242ms/step - loss: 32.9296 - root_mean_squared_error: 5.7384
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 6.1112 - root_mean_squared_error: 2.4721 - val_loss: 1.2392 - val_root_mean_squared_error: 1.1132

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.0515 - root_mean_squared_error: 1.0254
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.9357 - root_mean_squared_error: 0.9673 - val_loss: 0.8037 - val_root_mean_squared_error: 0.8965

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7363 - root_mean_squared_error: 0.8581
46/46 ━━━━━━━━━━━━━━━━━━━━

2026/05/26 13:08:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run welcoming-finch-251 at: http://127.0.0.1:5000/#/experiments/2/runs/e8ad014f515f4aa9ac4e0df85ede29eb

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 10s 239ms/step - loss: 31.3876 - root_mean_squared_error: 5.6025
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 8.3566 - root_mean_squared_error: 2.8908 - val_loss: 2.4347 - val_root_mean_squared_error: 1.5603

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.8181 - root_mean_squared_error: 1.3484
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.9193 - root_mean_squared_error: 1.3854 - val_loss: 1.7925 - val_root_mean_squared_error: 1.3388

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.2880 - root_mean_squared_error: 1.1349
46/46

...

🏃 View run bustling-doe-199 at: http://127.0.0.1:5000/#/experiments/2/runs/8469a2e74b8946e9b8524bf060db3e91

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 12s 268ms/step - loss: 36.7490 - root_mean_squared_error: 6.0621
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.1672 - root_mean_squared_error: 2.0414 - val_loss: 1.4791 - val_root_mean_squared_error: 1.2162

Epoch 2/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.3529 - root_mean_squared_error: 1.1632
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 834us/step - loss: 0.8656 - root_mean_squared_error: 0.9304 - val_loss: 0.8173 - val_root_mean_squared_error: 0.9041

Epoch 3/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.1182 - root_mean_squared_error: 1.0575
46/46 

2026/05/26 13:09:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run serious-sloth-955 at: http://127.0.0.1:5000/#/experiments/2/runs/dbb7358031a04b44be84eb94325af345

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

100%|██████████| 8/8 [00:32<00:00,  4.11s/trial, best loss: 0.7366483807563782]

2026/05/26 13:09:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best parameters: {'lr': np.float64(0.06090071435010334), 'momentum': np.float64(0.9010114985039333)}
Best eval rmse: 0.7366483807563782
🏃 View run marvelous-bear-516 at: http://127.0.0.1:5000/#/experiments/2/runs/bd6c693a3aee4996b84c725dbf1c6396
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


Si estamos contentos con un modelo en concreto podemos proceder a registrarlo:

![registry](img/mlflowreg.png)

## Exponer modelo

MLFlow serving: https://mlflow.org/docs/latest/ml/deployment/

![serving](https://mlflow.org/docs/latest/assets/images/mlflow-deployment-overview-99db410b2c58fedf506eb9ce5aa41a86.png)

Una vez hecho esto es sencillo invocar al proceso que sirve el modelo desde la terminal. Para ello es necesario establecer la URL del servidor de tracking en una variable local previamente:

```
export MLFLOW_TRACKING_URI=http://localhost:5000
```

Puede que para la gestión del entorno os pida también incluir las librerías [pyenv](https://github.com/pyenv/pyenv) y virtualenv (`!pip install virtualenv`).

Una vez configurada vuestra máquina, se vuelve un proceso sencillo en el que poder invocar el comando siguiente para servir el modelo:

```
mlflow models serve -m "models:/<nombre del modelo>/1" --port 5002
```

In [20]:
import requests

url_modelo = "http://localhost:5002/invocations"

json_data = {"dataframe_split": {
                "columns": [
                    "fixed acidity","volatile acidity","citric acid","residual sugar","chlorides","free sulfur dioxide","total sulfur dioxide","density","pH","sulphates","alcohol"],
                    "data": [[7,0.27,0.36,20.7,0.045,45,170,1.001,3,0.45,8.8]]}
}
headers = {'Content-Type' : 'application/json'}

response = requests.post(url=url_modelo, headers=headers, json=json_data)
print(response.status_code)

ConnectionError: HTTPConnectionPool(host='localhost', port=5002): Max retries exceeded with url: /invocations (Caused by NewConnectionError("HTTPConnection(host='localhost', port=5002): Failed to establish a new connection: [Errno 61] Connection refused"))

In [ ]:
response.content